Bonus Code for Chapter 5
Alternative Weight Loading from PyTorch state dicts

In [ ]:
# 从 importlib.metadata 导入 version 函数，用于查询已安装包的版本号
from importlib.metadata import version

# 这里只需要检查 torch 的版本
# （后面会用到 torch.load(..., weights_only=True) 等较新版本才稳定支持的特性）
pkgs = ["torch"]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# 定义 GPT 模型的基础配置（与原书 ch04 中 GPTModel 所需的配置字典保持一致）
# 注意：这里的字段名（vocab_size、context_length、drop_rate、qkv_bias、emb_dim、
# n_layers、n_heads）必须与 GPTModel 内部使用的名称一致，
# 否则模型结构会和待加载的 state_dict 权重形状对不上
BASE_CONFIG = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "drop_rate": 0.0,       # Dropout rate
    "qkv_bias": True        # Query-key-value bias
}

# 四种不同规模 GPT-2 模型各自的超参数（嵌入维度、层数、注意力头数）
# 这些数值来自 OpenAI 官方发布的 GPT-2 模型规格
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}


# 选择要使用的模型规格，并用对应配置更新 BASE_CONFIG
# （之后构建 GPTModel 时会用这个合并后的配置字典）
CHOOSE_MODEL = "gpt2-small (124M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [ ]:
# 指定要下载/加载的 PyTorch 原生权重文件名（.pth 格式，即 torch.save 保存的 state_dict）
# 这些文件是作者预先用本仓库的 GPTModel 结构训练/转换好并上传到 HuggingFace 的，
# 因此权重里的键名与 GPTModel 的参数名是完全对应的，不需要像加载 OpenAI 官方
# TensorFlow checkpoint 那样做键名映射
file_name = "gpt2-small-124M.pth"
# file_name = "gpt2-medium-355M.pth"
# file_name = "gpt2-large-774M.pth"
# file_name = "gpt2-xl-1558M.pth"

In [ ]:
import os
import requests

# 拼接 HuggingFace Hub 上对应权重文件的下载地址
url = f"https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/{file_name}"

# 如果本地还没有这个权重文件，就通过 HTTP 下载并保存到当前目录
if not os.path.exists(file_name):
    response = requests.get(url, timeout=60)
    response.raise_for_status()  # 请求失败（如 404）时抛出异常，避免静默写入错误内容
    with open(file_name, "wb") as f:
        f.write(response.content)
    print(f"Downloaded to {file_name}")

In [ ]:
import torch
# 原代码此处错误地写成了 `from Build_an_LLM_from_Scratch.ch04 import GPTModel`，
# 但该模块实际的包名是 `llms_from_scratch`（并非本仓库文件夹名），
# 因此这里修正为正确的包名，否则会因找不到模块而导入失败
from llms_from_scratch.ch04 import GPTModel
# For llms_from_scratch installation instructions, see:
# https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg


# 先按配置实例化一个"空壳" GPTModel，其内部各层参数名（如
# tok_emb.weight、trf_blocks.0.att.W_query.weight 等）
# 是由 GPTModel 的定义（nn.Module 中各子模块的属性名）决定的
gpt = GPTModel(BASE_CONFIG)
# torch.load 读取 .pth 文件得到一个 state_dict（键名 -> 张量 的字典）
# weights_only=True 表示只反序列化张量数据，不执行任意 pickle 代码，更安全
# 因为这个 .pth 文件本身就是用同一套 GPTModel 结构 torch.save 导出的，
# 所以 state_dict 里的键名与 gpt.state_dict() 的键名完全一致，
# load_state_dict 可以直接按键名逐一赋值，不需要额外的键名映射/转换逻辑
gpt.load_state_dict(torch.load(file_name, weights_only=True))
gpt.eval()  # 切换到推理模式（关闭 dropout 等训练专用行为）

# 有 GPU（CUDA）就用 GPU，否则退回 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpt.to(device);

In [ ]:
import tiktoken
# 同样修正错误的包名：应从 llms_from_scratch 而不是 Build_an_LLM_from_Scratch 导入
from llms_from_scratch.ch05 import generate, text_to_token_ids, token_ids_to_text


torch.manual_seed(123)  # 固定随机种子，保证生成结果可复现

tokenizer = tiktoken.get_encoding("gpt2")  # 加载 GPT-2 使用的 BPE 分词器

# 用刚刚加载好官方权重的模型做一次文本生成，
# 如果权重加载（含键名对应关系）是正确的，输出应该是通顺、有意义的英文续写，
# 而不是乱码，这也是验证权重加载是否成功的一种直观方式
token_ids = generate(
    model=gpt.to(device),
    idx=text_to_token_ids("Every effort moves", tokenizer).to(device),
    max_new_tokens=30,
    context_size=BASE_CONFIG["context_length"],
    top_k=1,          # 每步只取概率最高的 token（贪心解码）
    temperature=1.0   # 温度为 1.0，配合 top_k=1 时实际上等价于纯贪心解码
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

In [ ]:
# 接下来演示另一种权重文件格式：safetensors
# safetensors 是 HuggingFace 推出的一种更安全、加载更快的张量序列化格式，
# 相比 .pth（基于 pickle）不会执行任意代码，也支持内存映射（mmap）快速加载
# 这里同样只是切换文件名，键名映射方式与上面 .pth 方式完全一样
file_name = "gpt2-small-124M.safetensors"
# file_name = "gpt2-medium-355M.safetensors"
# file_name = "gpt2-large-774M.safetensors"
# file_name = "gpt2-xl-1558M.safetensors"

In [ ]:
import os
import requests

# 拼接对应 safetensors 权重文件在 HuggingFace Hub 上的下载地址
url = f"https://huggingface.co/rasbt/gpt2-from-scratch-pytorch/resolve/main/{file_name}"

# 本地不存在则下载保存
if not os.path.exists(file_name):
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    with open(file_name, "wb") as f:
        f.write(response.content)
    print(f"Downloaded to {file_name}")

In [ ]:
# Load file

# safetensors 提供的 load_file 函数：直接从 .safetensors 文件读取出
# 一个普通的 dict（键名 -> 张量），格式与 torch.load 得到的 state_dict 兼容
from safetensors.torch import load_file

# 重新实例化一个空的 GPTModel（结构、键名与之前完全一样）
gpt = GPTModel(BASE_CONFIG)
# 因为 safetensors 文件里的键名同样是按 GPTModel 的参数名保存的，
# 所以依然可以直接 load_state_dict，无需任何键名转换/映射
gpt.load_state_dict(load_file(file_name))
gpt.eval();  # 切换到推理模式

In [ ]:
# 用通过 safetensors 加载权重的模型再次生成文本，
# 复用前面已经创建好的 tokenizer 和随机种子设置
# 如果输出与前面用 .pth 加载时的结果一致，说明两种格式加载到的权重是相同的，
# 也进一步验证了 safetensors 方式的键名映射同样正确
token_ids = generate(
    model=gpt.to(device),
    idx=text_to_token_ids("Every effort moves", tokenizer).to(device),
    max_new_tokens=30,
    context_size=BASE_CONFIG["context_length"],
    top_k=1,
    temperature=1.0
)

print("Output text:\n", token_ids_to_text(token_ids, tokenizer))